In [ ]:
!pip install -q faiss-cpu sentence-transformers transformers

# 1 Data Loading

In [ ]:
import os, pickle, re, json
import numpy as np, pandas as pd, faiss, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm import tqdm

BASE = '/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag'
WORKDIR = '/kaggle/working'
CACHE_DIR = os.path.join(WORKDIR, 'cache')
os.makedirs(CACHE_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SKIP_HASHES = {'7d49c038'}

TOP_K_FAISS = 100
TOP_K_RERANK = 10
TOP_K_FINAL = 5

MAX_PER_VIDEO = 2
OVERLAP_THRESHOLD = 0.5

USE_CLIP = False
CLIP_WEIGHT = 0.2
CLIP_NUM_FRAMES = 12

print(f'Device: {DEVICE}')


# 2 Chunk Generation

In [ ]:
def extract_hash(path):
    m = re.search(r'_([a-f0-9]+)[\.\.\w]*$', str(path))
    return m.group(1) if m else None

# Keep for backward compatibility with the previous notebook behavior.
def merge_and_shrink(candidates, gap=10.0, shrink=0.7):
    by_video = {}
    for c in candidates:
        by_video.setdefault(c['video_hash'], []).append(c)
    merged = []
    for vh, chks in by_video.items():
        chks = sorted(chks, key=lambda x: x['start'])
        cur, best = chks[0].copy(), chks[0].copy()
        for nxt in chks[1:]:
            if nxt['start'] <= cur['end'] + gap:
                cur['end'] = max(cur['end'], nxt['end'])
                if nxt['score'] > cur['score']:
                    cur['score'] = nxt['score']; best = nxt.copy()
            else:
                bc = (best['start'] + best['end']) / 2
                bh = (best['end'] - best['start']) / 2
                ch = (cur['end'] - cur['start']) / 2
                nh = ch * (1 - shrink) + bh * shrink
                cur['start'] = max(cur['start'], bc - nh)
                cur['end'] = min(cur['end'], bc + nh)
                merged.append(cur)
                cur, best = nxt.copy(), nxt.copy()
        bc = (best['start'] + best['end']) / 2
        bh = (best['end'] - best['start']) / 2
        ch = (cur['end'] - cur['start']) / 2
        nh = ch * (1 - shrink) + bh * shrink
        cur['start'] = max(cur['start'], bc - nh)
        cur['end'] = min(cur['end'], bc + nh)
        merged.append(cur)
    merged.sort(key=lambda x: x['score'], reverse=True)
    return merged

# Multi-scale chunking (30s, 60s). Store segment index range for temporal refinement.
def make_chunks(segments, window, step):
    chunks = []
    if not segments:
        return chunks
    t = segments[0]['start']
    while t < segments[-1]['end']:
        idxs = [i for i, s in enumerate(segments) if s['end'] > t and s['start'] < t + window]
        if idxs:
            i0, i1 = idxs[0], idxs[-1]
            ws = segments[i0:i1+1]
            chunks.append({
                'start': ws[0]['start'],
                'end': ws[-1]['end'],
                'text': ' '.join(s['text'].lower().strip() for s in ws),
                'seg_start_idx': i0,
                'seg_end_idx': i1,
            })
        t += step
    return chunks


In [ ]:
with open(f'{BASE}/transcripts.pkl', 'rb') as f:
    transcripts = pickle.load(f)

h2f = {}
h2path = {}
for p in pd.read_csv(f'{BASE}/video_files.csv')['video_path']:
    h = extract_hash(p)
    if h:
        h2f[h] = re.sub(r'\.\w+$', '', p.split('/')[-1])
        h2path[h] = f"{BASE}/{p}"

# Build hash->segments lookup for temporal refinement.
transcripts_by_hash = {}
for key, segs in transcripts.items():
    vh = extract_hash(key)
    if vh:
        transcripts_by_hash[vh] = segs

test = pd.read_csv(f'{BASE}/test/test.csv')
print(f'Test: {len(test)} queries, Videos: {len(h2f)}')


In [ ]:
all_chunks = []
for key, segs in tqdm(transcripts.items()):
    vh = extract_hash(key)
    if not vh or vh in SKIP_HASHES:
        continue
    for w, s in [(30.0, 15.0), (60.0, 30.0)]:
        for ch in make_chunks(segs, w, s):
            ch['video_hash'] = vh
            all_chunks.append(ch)
print(f'Chunks: {len(all_chunks)}')


# 3 Query Expansion

In [ ]:
def expand_query(query, max_expansions=6):
    q = str(query).lower().strip()
    if not q:
        return [q]

    # Simple phrase-level replacements.
    phrase_map = {
        'refrigerator': ['fridge'],
        'open the': ['look inside the'],
        'open': ['look inside', 'unlock'],
        'take': ['remove', 'grab'],
        'pick up': ['lift', 'grab'],
        'put down': ['place', 'set down'],
        'turn on': ['switch on', 'power on'],
        'turn off': ['switch off', 'power off'],
    }

    expansions = {q}

    # Phrase substitutions.
    for k, reps in phrase_map.items():
        if k in q:
            for r in reps:
                expansions.add(q.replace(k, r))

    # Token-level synonyms (limited to avoid explosion).
    token_map = {
        'refrigerator': ['fridge'],
        'cupboard': ['cabinet'],
        'bottle': ['flask'],
        'sofa': ['couch'],
        'television': ['tv'],
        'shelf': ['rack'],
        'backpack': ['bag'],
        'laptop': ['notebook'],
    }

    tokens = q.split()
    for i, tok in enumerate(tokens):
        if tok in token_map:
            for r in token_map[tok]:
                nt = tokens.copy()
                nt[i] = r
                expansions.add(' '.join(nt))

    # Heuristic paraphrases.
    if q.startswith('open '):
        expansions.add(q.replace('open ', 'look inside ', 1))
    if q.startswith('close '):
        expansions.add(q.replace('close ', 'shut ', 1))

    out = list(expansions)
    # Keep the original query first.
    if q in out:
        out.remove(q)
        out = [q] + out
    return out[:max_expansions]


# 4 Embedding Generation

In [ ]:
model = SentenceTransformer('olegGerbylev/bge-m3-video-retrieval-ft', device=DEVICE, trust_remote_code=True)

chunk_cache_path = os.path.join(CACHE_DIR, 'chunk_emb_bge_m3.npy')

if os.path.exists(chunk_cache_path):
    emb = np.load(chunk_cache_path)
    if emb.shape[0] != len(all_chunks):
        emb = None
else:
    emb = None

if emb is None:
    emb = model.encode(
        [ch['text'] for ch in all_chunks],
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    np.save(chunk_cache_path, emb)

print(f'Chunk embeddings: {emb.shape}')

# On-demand cache for per-video segment embeddings used in temporal refinement.
segment_emb_cache = {}

def get_segment_embeddings(video_hash):
    if video_hash in segment_emb_cache:
        return segment_emb_cache[video_hash]

    cache_path = os.path.join(CACHE_DIR, f'seg_emb_{video_hash}.npy')
    if os.path.exists(cache_path):
        seg_emb = np.load(cache_path)
    else:
        segs = transcripts_by_hash[video_hash]
        texts = [s['text'].lower().strip() for s in segs]
        seg_emb = model.encode(
            texts,
            batch_size=64,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype('float32')
        np.save(cache_path, seg_emb)
    segment_emb_cache[video_hash] = seg_emb
    return seg_emb


# 5 FAISS Index

In [ ]:
dim = emb.shape[1]

# HNSW index for faster approximate search.
try:
    index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
except TypeError:
    # Fallback for older faiss versions without metric argument.
    index = faiss.IndexHNSWFlat(dim, 32)

index.hnsw.efConstruction = 200
index.hnsw.efSearch = max(100, TOP_K_FAISS * 2)

index.add(emb)
print(f'Index: {index.ntotal} vectors, dim={dim}')


# 6 Retrieval Pipeline

In [ ]:
def encode_query(query):
    expanded = expand_query(query)
    q_emb = model.encode(
        expanded,
        batch_size=16,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    qv = q_emb.mean(axis=0, keepdims=True)
    # Normalize after averaging to keep cosine/IP consistency.
    qv /= np.linalg.norm(qv, axis=1, keepdims=True) + 1e-12
    return qv, expanded

query_cache = {}

def retrieve_candidates(query):
    q = str(query).lower().strip()
    if q in query_cache:
        qv = query_cache[q]
    else:
        qv, _ = encode_query(q)
        query_cache[q] = qv
    scores, ids = index.search(qv, TOP_K_FAISS)
    cands = []
    for s, i in zip(scores[0], ids[0]):
        if i == -1:
            continue
        ch = all_chunks[i]
        cands.append({
            'video_hash': ch['video_hash'],
            'start': ch['start'],
            'end': ch['end'],
            'text': ch['text'],
            'seg_start_idx': ch['seg_start_idx'],
            'seg_end_idx': ch['seg_end_idx'],
            'faiss_score': float(s),
        })
    return qv, cands


# 7 Reranking

In [ ]:
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=DEVICE)

def rerank(query, candidates, batch_size=32, top_k=TOP_K_RERANK):
    if not candidates:
        return []
    pairs = [(query, c['text']) for c in candidates]
    scores = reranker.predict(pairs, batch_size=batch_size, show_progress_bar=False)
    for c, s in zip(candidates, scores):
        c['rerank_score'] = float(s)
        c['score'] = float(s)
    candidates.sort(key=lambda x: x['score'], reverse=True)
    return candidates[:top_k]


# 8 Temporal Refinement

In [ ]:
def refine_timestamp(query_emb, cand):
    vh = cand['video_hash']
    segs = transcripts_by_hash.get(vh)
    if not segs:
        return cand
    i0, i1 = cand['seg_start_idx'], cand['seg_end_idx']
    i0 = max(0, i0)
    i1 = min(len(segs) - 1, i1)

    seg_emb = get_segment_embeddings(vh)
    seg_slice = seg_emb[i0:i1+1]
    if seg_slice.size == 0:
        return cand

    sims = seg_slice @ query_emb[0]
    best_idx = int(np.argmax(sims))
    best_seg = segs[i0 + best_idx]

    cand['start'] = float(best_seg['start'])
    cand['end'] = float(best_seg['end'])
    cand['segment_score'] = float(sims[best_idx])
    return cand


# 9 Optional CLIP Fusion

In [ ]:
clip_model = None
clip_processor = None
clip_cache = {}

if USE_CLIP:
    from transformers import CLIPModel, CLIPProcessor
    try:
        import cv2
    except Exception as e:
        raise RuntimeError('OpenCV is required for USE_CLIP=True') from e

    clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
    clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

    def _sample_frames_cv2(video_path, num_frames=CLIP_NUM_FRAMES):
        cap = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            return []
        idxs = np.linspace(0, total - 1, num_frames, dtype=int)
        frames = []
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            if not ok:
                continue
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        return frames

    def get_clip_video_emb(video_hash):
        if video_hash in clip_cache:
            return clip_cache[video_hash]

        cache_path = os.path.join(CACHE_DIR, f'clip_vid_{video_hash}.npy')
        if os.path.exists(cache_path):
            vemb = np.load(cache_path)
        else:
            vpath = h2path.get(video_hash)
            if not vpath or not os.path.exists(vpath):
                return None
            frames = _sample_frames_cv2(vpath, CLIP_NUM_FRAMES)
            if not frames:
                return None
            inputs = clip_processor(images=frames, return_tensors='pt')
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.no_grad():
                feats = clip_model.get_image_features(**inputs)
            feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
            vemb = feats.mean(dim=0).cpu().numpy().astype('float32')
            vemb = vemb / (np.linalg.norm(vemb) + 1e-12)
            np.save(cache_path, vemb)

        clip_cache[video_hash] = vemb
        return vemb

    def get_clip_text_emb(query):
        inputs = clip_processor(text=[query], return_tensors='pt', padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            tfeat = clip_model.get_text_features(**inputs)
        tfeat = tfeat / (tfeat.norm(dim=-1, keepdim=True) + 1e-12)
        return tfeat[0].cpu().numpy().astype('float32')

else:
    def get_clip_video_emb(video_hash):
        return None

    def get_clip_text_emb(query):
        return None


# 10 Submission Generation

In [ ]:
def overlap_ratio(a, b):
    inter = max(0.0, min(a['end'], b['end']) - max(a['start'], b['start']))
    if inter <= 0:
        return 0.0
    la = max(1e-6, a['end'] - a['start'])
    lb = max(1e-6, b['end'] - b['start'])
    return inter / min(la, lb)


def diversity_filter(candidates, top_k=TOP_K_FINAL, max_per_video=MAX_PER_VIDEO, overlap_thr=OVERLAP_THRESHOLD):
    selected = []
    per_video = {}
    for c in sorted(candidates, key=lambda x: x['score'], reverse=True):
        vh = c['video_hash']
        if per_video.get(vh, 0) >= max_per_video:
            continue
        too_close = False
        for s in selected:
            if s['video_hash'] == vh and overlap_ratio(c, s) > overlap_thr:
                too_close = True
                break
        if too_close:
            continue
        selected.append(c)
        per_video[vh] = per_video.get(vh, 0) + 1
        if len(selected) >= top_k:
            break
    return selected


results = []
for _, row in tqdm(test.iterrows(), total=len(test)):
    query = str(row['question']).lower().strip()

    qv, cands = retrieve_candidates(query)

    # Rerank and keep top candidates.
    reranked = rerank(query, cands, top_k=TOP_K_RERANK)

    # Temporal refinement on reranked list.
    refined = [refine_timestamp(qv, c.copy()) for c in reranked]

    # Optional CLIP fusion at video level.
    if USE_CLIP and refined:
        q_clip = get_clip_text_emb(query)
        for c in refined:
            vemb = get_clip_video_emb(c['video_hash'])
            if vemb is None:
                c['clip_score'] = 0.0
            else:
                c['clip_score'] = float(np.dot(q_clip, vemb))
            c['score'] = (1 - CLIP_WEIGHT) * c['score'] + CLIP_WEIGHT * c['clip_score']

    # Diversity filtering and final top-k selection.
    final_hits = diversity_filter(refined, top_k=TOP_K_FINAL)
    results.append({'query_id': row['query_id'], 'hits': final_hits})


rows = []
for r in results:
    d = {'query_id': r['query_id']}
    for rk in range(1, 6):
        if rk <= len(r['hits']):
            h = r['hits'][rk-1]
            d[f'video_file_{rk}'] = h2f.get(h['video_hash'], h['video_hash'])
            d[f'start_{rk}'] = round(h['start'], 1)
            d[f'end_{rk}'] = round(h['end'], 1)
        else:
            d[f'video_file_{rk}'], d[f'start_{rk}'], d[f'end_{rk}'] = '', 0.0, 0.0
    rows.append(d)

cols = ['query_id']
for rk in range(1, 6):
    cols += [f'video_file_{rk}', f'start_{rk}', f'end_{rk}']

sub = pd.DataFrame(rows, columns=cols)
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub.shape)
sub.head()
